# XGBoost Classifier on Baseline Dataset
We use XGBoost to predict Sepsis from vital signal, lab test, and demographics data.

## Assistant Functions

In [ ]:
def concat_patients(train_dir, patient_list):
    """
    Concatenate individual patient dataframes for
    the training, valid, or other dataframes.
    """
    p = pd.read_csv(train_dir + '/' + patient_list[0], sep="|")
    p['patient_id'] = patient_list[0][1:7]
    for i in range(1, len(patient_list)):
        p_n = pd.read_csv(train_dir + '/' + patient_list[i], sep="|")
        p_n['patient_id'] = patient_list[i][1:7]
        p = pd.concat([p, p_n])
    return p

def predictors_labels_allocator(df):
    """Allocate predictors and labels."""
    col_names = df.columns
    X = np.array(df[col_names[:-2]].values)
    y = df[col_names[-2]].values
    return X, y

def F(beta, precision, recall):
    """Calculate f-beta score."""
    return (beta*beta + 1)*precision*recall / (beta*beta*precision + recall)

## Load the data
Import standard modules.

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import joblib

from sklearn import preprocessing
from sklearn.metrics import (precision_recall_curve, auc, roc_curve,
                             precision_score, recall_score, accuracy_score)
from sklearn.model_selection import GridSearchCV
from plot_metric.functions import BinaryClassification
from xgboost import XGBClassifier

In [ ]:
train_dir = '../../Data/training_setA/baseline/train_baseline/'
valid_dir = '../../Data/training_setA/baseline/val_baseline/'
test_dir  = '../../Data/training_setA/baseline/test_baseline/'

TRAIN_PATH = train_dir + 'data_baseline_train.parquet'
VALID_PATH = valid_dir + 'data_baseline_valid.parquet'
TEST_PATH  = test_dir  + 'data_baseline_test.parquet'

tr_patients  = sorted(os.listdir(train_dir))
vld_patients = sorted(os.listdir(valid_dir))
ts_patients  = sorted(os.listdir(test_dir))

print('num training patients:', len(tr_patients))
print('num valid patients:',    len(vld_patients))
print('num test patients:',     len(ts_patients))

Concatenate patients into `train_df`, `valid_df`, and `test_df`.

In [ ]:
try:
    if not all(os.path.exists(p) for p in [TRAIN_PATH, VALID_PATH, TEST_PATH]):
        raise FileNotFoundError("One or more parquet files missing")
    train_df = pd.read_parquet(TRAIN_PATH)
    valid_df = pd.read_parquet(VALID_PATH)
    test_df  = pd.read_parquet(TEST_PATH)
    print("Loaded data from parquet files")
except Exception as e:
    print(e)
    print("Creating new parquet files...")
    train_df = concat_patients(train_dir, tr_patients)
    valid_df = concat_patients(valid_dir, vld_patients)
    test_df  = concat_patients(test_dir,  ts_patients)
    train_df.to_parquet(TRAIN_PATH, index=False)
    valid_df.to_parquet(VALID_PATH, index=False)
    test_df.to_parquet(TEST_PATH,  index=False)
    print("Saved data to parquet files")

Check missing data.

In [ ]:
print('train data has missing values:', train_df.isnull().sum().sum() != 0)
print('valid data has missing values:', valid_df.isnull().sum().sum() != 0)
print('test data has missing values:', test_df.isnull().sum().sum() != 0)

## XGBoost Classification for Sepsis
Separate features (X) and labels (y), then fit the model on training data.

In [ ]:
Xtr,  ytr  = predictors_labels_allocator(train_df)
Xvld, yvld = predictors_labels_allocator(valid_df)
Xts,  yts  = predictors_labels_allocator(test_df)

Define and fit a `StandardScaler` on training data.

In [ ]:
scaler = preprocessing.StandardScaler()
scaler.fit(Xtr)
Xtr  = scaler.transform(Xtr)
Xvld = scaler.transform(Xvld)
Xts  = scaler.transform(Xts)

Create a `XGBoost` model and fit on training data.

In [ ]:
model = XGBClassifier(n_estimators=100, learning_rate=0.1, random_state=42,
                      use_label_encoder=False, eval_metric='logloss', n_jobs=-1)
model.fit(Xtr, ytr, eval_set=[(Xvld, yvld)], verbose=False)

Measure validation accuracy.

In [ ]:
yhat = model.predict(Xvld)
acc = np.mean(yhat == yvld)
print('Accuracy on the validation data is {0:f}'.format(acc))

## Feature Importance

In [ ]:
importances = model.feature_importances_
indices = np.argsort(importances)[::-1]
feature_names = list(train_df.columns[:-2])

plt.figure(figsize=(12, 5))
plt.title("Top 20 Feature Importances – XGBoost")
plt.bar(range(20), importances[indices[:20]])
plt.xticks(range(20), [feature_names[i] for i in indices[:20]], rotation=90)
plt.tight_layout()
plt.show()

print("Top 4 most significant features:")
for k in range(4):
    print(f"  {k+1}. {feature_names[indices[k]]}")

## Hyperparameter Tuning
Use `GridSearchCV` to find optimal hyperparameters.

In [ ]:
from xgboost import XGBClassifier

model_tune = XGBClassifier(random_state=42, use_label_encoder=False,
                           eval_metric='logloss', n_jobs=-1)

hyperparameters = {
    'n_estimators':  [50, 100, 200],
    'learning_rate': [0.01, 0.1, 0.3],
    'max_depth':     [3, 6, 9],
}

In [ ]:
CLF_PATH        = "../../Data/xgb_tuning_clf.joblib"
BEST_MODEL_PATH = "../../Data/xgb_tuning_best_model.joblib"

try:
    if not all(os.path.exists(p) for p in [CLF_PATH, BEST_MODEL_PATH]):
        raise FileNotFoundError("One or more joblib files missing")
    clf        = joblib.load(CLF_PATH)
    best_model = joblib.load(BEST_MODEL_PATH)
    print("Loaded clf and best model from joblib files")
except Exception as e:
    print(e)
    print("Error opening files, creating new...")
    clf        = GridSearchCV(model_tune, hyperparameters, cv=5, verbose=1, n_jobs=-1)
    best_model = clf.fit(Xtr, ytr)
    joblib.dump(clf,        CLF_PATH)
    joblib.dump(best_model, BEST_MODEL_PATH)
    print("Saved clf and best model to joblib files")

In [ ]:
print(type(clf))

In [ ]:
best_model.best_params_

## Test Data Prediction and Performance

In [ ]:
# Predict on test set
yhat_ts      = best_model.predict(Xts)
yhat_probas  = best_model.predict_proba(Xts)[:, 1]

acc_ts = np.mean(yhat_ts == yts)
print('Accuracy on the test data is {0:f}'.format(acc_ts))

Plot ROC and PR curves.

In [ ]:
from plot_metric.functions import BinaryClassification
bc = BinaryClassification(yts, yhat_probas, labels=["nonSepsis", "Sepsis"])

plt.figure(figsize=(15, 10))
plt.subplot2grid(shape=(2, 6), loc=(0, 0), colspan=2)
bc.plot_roc_curve()
plt.subplot2grid((2, 6), (0, 3), colspan=2)
bc.plot_precision_recall_curve()
plt.show()

Classification report.

In [ ]:
precision, recall, _ = precision_recall_curve(yts, yhat_probas)
fpr, tpr, _          = roc_curve(yts, yhat_probas)

print('f1 score  {0:.4f}:'.format(F(1, np.mean(precision), np.mean(recall))))
print('f2 score  {0:.4f}:'.format(F(2, np.mean(precision), np.mean(recall))))
print('precision {0:.4f}:'.format(precision_score(yts, yhat_ts)))
print('recall    {0:.4f}:'.format(recall_score(yts, yhat_ts)))
print('AUPRC     {0:.4f}:'.format(auc(recall, precision)))
print('AUROC     {0:.4f}:'.format(auc(fpr, tpr)))
print('Acc       {0:.4f}:'.format(accuracy_score(yts, yhat_ts)))

bc.print_report()